# ALD_외부 주입 CSV — 예측·속도 확인

**공정: PEALD. 제공받은 주입 CSV 또는 새 CSV를 기존 GEVIS로 예측합니다.** 내부 합성변화 연구나 원본 Test 정확도 평가가 아닙니다. 정답 채점·새 주입·학습은 하지 않습니다.

기본은 1파일·최대8개 완성 창이며 모든 지원 센서의 세 답변을 반환합니다. 실제 장비 스트리밍이 아닌 CSV 재생입니다.


## 1. PEALD 입력 경로

이 파일은 PEALD 전용입니다. INPUT_DIRECTORY에 승인된 PEALD CSV 경로를 지정합니다. 다른 공정 CSV로 대신 실행하지 않습니다. PECVD 전처리 형식과 PEALD 원시 CSV는 다릅니다. 전체 재생은 두 MAX 설정을 None으로 바꿉니다.


In [ ]:
from pathlib import Path
import sys,os,importlib,inspect
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
ROOT=next(p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'src/pecvd').is_dir())
if str(ROOT/'src') not in sys.path:sys.path.insert(0,str(ROOT/'src'))
from IPython.display import display
from pecvd.experiments import external_test as external,pecvd_external as incoming,fast_experiments as fast,fast_assessment
from pecvd.experiments.fusion_answers import SYSTEM as GEVIS_SYSTEM
from pecvd.experiments.process_inputs import configured_process_path
from pecvd.process_config import load_process_config
for _adapter in (external,incoming):
    if Path(_adapter.__file__).resolve().parent != (ROOT/'src/pecvd/experiments').resolve():
        raise RuntimeError('Wrong checkout; restart this project kernel.')
    if 'max_windows_per_file' not in inspect.signature(_adapter.evaluate).parameters:
        importlib.reload(_adapter)
DOMAIN='PEALD'  # Fixed domain for this notebook; use the other prefixed notebook for the other process.
INPUT_DIRECTORY=ROOT/'data/injected_data'  # Supplied PEALD CSVs; no new injection is generated here.
MAX_FILES=1  # Set None explicitly to process all supplied files.
MAX_WINDOWS_PER_FILE=8  # Set None explicitly for all complete windows per file.
SELECTED_STEPS=None
SELECTED_SENSORS=None  # All supported sensors; preserve full context and three answers.
WINDOW_STRIDE=None  # Complete native windows, without overlapping row-by-row replay.
CENTER_BATCH_SIZE=8
SHOW_DETAILS=False


## 2. 기존 모델 로드

정확한 학습 가중치나 TRAIN 참조가 없으면 중단하거나 미처리 사유를 남깁니다. 임의 모델로 답을 대신하지 않습니다.


In [ ]:
if DOMAIN!='PEALD':
    raise ValueError('This ALD_ notebook accepts PEALD only; select the other process notebook.')
if MAX_FILES is not None and (type(MAX_FILES) is not int or MAX_FILES<1):
    raise ValueError('MAX_FILES must be a positive integer or None')
if not INPUT_DIRECTORY.is_dir():
    raise FileNotFoundError('PEALD supplied CSV directory is missing; set INPUT_DIRECTORY to approved PEALD inputs. Other-process CSVs are not substitutes.')
ALL_INPUT_PATHS=external.inventory(INPUT_DIRECTORY)
INPUT_PATHS=ALL_INPUT_PATHS if MAX_FILES is None else ALL_INPUT_PATHS[:MAX_FILES]
CONTRACT=external.load_peald(ROOT,steps=SELECTED_STEPS)
print({'experiment':'ALD external CSV prediction','domain':DOMAIN,'available_files':len(ALL_INPUT_PATHS),'selected_files':len(INPUT_PATHS),'max_windows_per_file':MAX_WINDOWS_PER_FILE,'scoring':False,'training':False})


## 3. 예측과 속도

모든 지원 센서의 BOOLEAN·CHOICE·ORDINAL을 반환합니다. 시간은 완성된 모델 창을 받은 뒤 전체 답변을 구성할 때까지이며 파일 읽기·창 분리·장비 수집·큐·전송은 제외합니다. 정답 없는 입력에는 정확도·F1을 붙이지 않습니다.


In [ ]:
REPORT = external.evaluate(CONTRACT, INPUT_PATHS, reference_mode='train_bank', score_original=False, centers=SELECTED_SENSORS, stride=WINDOW_STRIDE, max_windows_per_file=MAX_WINDOWS_PER_FILE, center_batch_size=CENTER_BATCH_SIZE, progress=print)
PREDICTIONS=REPORT.details;SUMMARY=REPORT.summary
print('ALD / PEALD: predictions and request timing only; no accuracy scoring')
FAST_ASSESSMENT=fast_assessment.compact(REPORT,display,evidence_scope=DOMAIN+'_input_preview',
    predictions=18,show_details=SHOW_DETAILS)
display(REPORT.diagnostics['coverage'][['case_id','available_windows','windows','windows_not_requested_by_limit']])


전체 예측은 PREDICTIONS, 처리 범위·실패 사유는 REPORT.diagnostics에서 확인합니다. 기본 표는 일부 답변이며 SHOW_DETAILS=True로 상세 표를 봅니다. 빠르게 표시됐다는 사실이 평균50ms 달성이나 실제 고장 정확도를 증명하지는 않습니다.
